## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [41]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

MessageError: Error: credential propagation was unsuccessful

Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1 Exploratory Data Analysis

In [ ]:
import os as os
import duckdb
import plotly.express as px

In [ ]:
# defino los parametros
PARAM = {'experimento':'EDA-101',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [ ]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.1 Creacion de tablas

Para el Exploratory Data Analysis utilizo DuckDB
<br>Cargo los archivos a utilizar en TABLAS de DuckDB

In [ ]:
con = duckdb.connect()

In [ ]:
# creo la tabla del sell-in  transformando el campo periodo a tipo DATE

con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id,
           product_id,
           plan_precios_cuidados,
           cust_request_qty,
           cust_request_tn,
           tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

In [ ]:
# creo la tabla que posee la definicion de los productos

con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT *
    FROM read_csv_auto('/content/datasets/tb_productos.txt')
    ORDER BY product_id
""")

## 1.2 EDA Ventas Totales

In [ ]:
# creo las ventas por producto

con.execute("""
    CREATE OR REPLACE TABLE tb_ventas_global AS
    SELECT periodo,
           SUM(tn) tn
    FROM  tb_sellin
    GROUP BY periodo
    ORDER BY periodo
""")



In [ ]:
con.sql("""
SELECT  year(periodo) yean,
        round( SUM(tn) ) tn
FROM    tb_ventas_global
GROUP BY year(periodo)
ORDER BY year(periodo)
""").show()

¿Cómo interpreta lo que viene sucediendo año a año con las ventas?

In [ ]:
# Grafico global de ventas
gra = px.line(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales",
    labels={"periodo": "Periodo", "tn": "Toneladas"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [ ]:
# Grafico global de ventas con tendencia
gra = px.scatter(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales con tendencia",
    labels={"periodo": "Periodo", "tn": "Toneladas"},
    trendline='ols',
    trendline_color_override='red'
)

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

¿Cómo es la tendencia de las ventas en los últimos 3 años?

## 1.3  EDA Clientes
¿Cuánto market share tiene los clientes?

In [ ]:
# Creacion de tabla de clientes de 2019
con.execute("""
CREATE OR REPLACE TABLE tb_clientes_2019 AS
SELECT  customer_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' AND '2019-12-01'
GROUP BY customer_id
ORDER BY tn DESC
""")

In [ ]:
gra = px.bar(
    con.sql("SELECT tn_pct FROM tb_clientes_2019").df(),
    y="tn_pct",
    title="Densidad de tn por cliente",
    labels={ "tn_pct": "Toneladas Porcentaje"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [ ]:
# Cantidad de clientes distintos en 2019

con.sql("""
SELECT  COUNT( DISTINCT customer_id )
FROM    tb_clientes_2019
""")

In [ ]:
# el market share acumulado de los mejores clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn DESC
LIMIT 20
""").show()

Los primeros 13 productos representan el 50% de las ventas !

In [ ]:
# el market share acumulado de los PEORES clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn ASC
LIMIT 300
""").show()

Los 300 clientes menos importantes representan MENOS del 3% de las toneladas vendidas

In [ ]:
# evolucion de algunos clientes

clientes=[10001, 10002, 10012]

gra=px.line(
    con.sql(f"SELECT customer_id, periodo, SUM(tn) tn FROM tb_sellin WHERE customer_id in {clientes} GROUP BY customer_id, periodo ORDER BY 1,2").df(),
    x="periodo",
    y="tn",
    color='customer_id',
    title='customer_id =' + str(clientes),
    labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

## 1.4 EDA productos individuales
Market shared de los productos

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE tb_productos_201912 AS
SELECT  product_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' and '2019-12-01'
GROUP BY product_id
ORDER BY tn DESC
""")

In [ ]:
con.sql("""
SELECT *
FROM   tb_productos_201912
""").show()

In [ ]:
con.sql("""
SELECT  product_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_productos_201912
ORDER BY tn DESC
""").show()

In [ ]:
def graficar_un_producto(producto):
  gra = px.scatter(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id={producto} GROUP BY periodo").df(),
      x="periodo",
      y="tn",
      title=con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0],
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_traces(mode='lines')
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [ ]:
def graficar_productos(productos):
    for prod in productos:
      graficar_un_producto(prod)

In [ ]:
# gnero graficos INDEPENDIENTES de productos

graficar_productos( [20001, 20002, 20003, 20004])

## 1.5  EDA multiples productos

In [ ]:
productos = [ 20001, 20002]

tbl=con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1, 2").df()
display(tbl)

In [ ]:
def graficar_multiples_productos(productos):
  gra = px.line(
      con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1,2").df(),
      x="periodo",
      y="tn",
      color="product_id",
      title="Multiples Productos",
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [ ]:
def graficar_union_productos(productos):
  gra = px.line(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY periodo ORDER BY 1").df(),
      x="periodo",
      y="tn",
      title="Productos in " + str(productos),
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [ ]:
graficar_multiples_productos( [20001, 20002, 20003] )

In [ ]:
graficar_union_productos( [20001, 20002, 20003] )

## 1.6  EDA Estacionalidad Mayonesas

In [ ]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Mayonesa'")

In [ ]:
algunas_mayonesas = [20003, 20004, 20005]

In [ ]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {productos}")

In [ ]:
graficar_multiples_productos(algunas_mayonesas)

In [ ]:
graficar_union_productos(algunas_mayonesas)

¿Qué estacionalidad se observa para las mayonesas?

## 1.7 EDA Estacionalidad Sopas
Dado que entre que usualmente hay DOS MESES entre que el caminón sale por el portón de la planta de La Multinacional y es escaneado en la linea de caja del supermercado
<br> ¿En qué mes las familias empiezan a compar mas sopas?
<br> ¿En qué mes la multinacional vende más sopas?

In [ ]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Sopas'")

In [ ]:
algunas_sopas=[20234, 20265, 20302]

In [ ]:
graficar_multiples_productos(algunas_sopas)

In [ ]:
graficar_union_productos(algunas_sopas)

## 1.8 EDA Productos Infantiles
¿Qué forma tiene el sell-out de los productos nuevos?
<br>¿Los productos nuevos, canibalizan a otros productos de la misma familia de productos de La Multinacional?

In [ ]:
mostazas_infantes=[21144, 21146, 21154]

In [ ]:
graficar_multiples_productos(mostazas_infantes)

In [ ]:
mostazas_varias=[21144, 21146, 21154, 20884]

In [ ]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {mostazas_varias}")

In [ ]:
graficar_multiples_productos(mostazas_varias)

Podria pasar que para 2019-09 el product_id=20884 se vendió menos porque fue canibalizado por los nuevos tres productos?

## 1.9 EDA Productos discontinuados

¿Los productos discontinuados son reemplazados por otros que toman su lugar?

In [ ]:
mayonesas_discontinuadas=[20494, 20554]

In [ ]:
graficar_multiples_productos(mayonesas_discontinuadas)

In [ ]:
mayonesas=[20494, 20554, 20580]

In [ ]:
graficar_multiples_productos(mayonesas)